# Ultron voice - resumable training (no Google Drive)
Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom.

**How resuming works:** the last cell downloads `ultron_project_next.zip` (code + data + checkpoints). Next session, upload *that* file in cell 1 instead of the original, raise the `*_STEPS` numbers in cell 3, and run again. **Download it before closing the tab - Colab wipes its disk when the session ends.**

In [ ]:
import os
BASE = '/content/ultron'
if not os.path.exists(f'{BASE}/vtts'):
    from google.colab import files
    up = files.upload()   # ultron_project_v2.zip the first time, ultron_project_next.zip on later sessions
    name = list(up)[0]
    !mkdir -p {BASE} && unzip -q -o "{name}" -d {BASE}
    !rm -f "{name}"
!ls {BASE} {BASE}/runs/ultron
!pip -q install soundfile g2p_en
import nltk
for p in ['averaged_perceptron_tagger','averaged_perceptron_tagger_eng','cmudict']: nltk.download(p, quiet=True)
import torch; print(torch.cuda.get_device_name(0))

### Cell 2: train (~1 hour, stops and saves by itself)
`*_STEPS` are **total** targets. The acoustic model is at step 8000 now. Next session raise them (e.g. 20000 / 16000).

In [ ]:
import os
BASE = '/content/ultron'   # (re)defined here so this cell works even after a runtime restart
assert os.path.exists(f'{BASE}/vtts'), 'Project files are gone (runtime restarted) - run cell 1 again and upload the zip.'
import subprocess, time
ACOUSTIC_STEPS = 14000
VOCODER_STEPS  = 8000
MINUTES = 55
env = f'cd {BASE} && PYTHONPATH={BASE}'
init = '--init runs/ultron/vocoder_init.pt' if os.path.exists(f'{BASE}/runs/ultron/vocoder_init.pt') else ''
ac = subprocess.Popen(f'{env} python -u -m vtts train-acoustic --data data/all --out runs/ultron --steps {ACOUSTIC_STEPS} --amp --max-minutes {MINUTES} > ac.log 2>&1', shell=True)
vo = subprocess.Popen(f'{env} python -u -m vtts train-vocoder --data data/all --out runs/ultron --steps {VOCODER_STEPS} --small {init} --max-minutes {MINUTES} > voc.log 2>&1', shell=True)
while ac.poll() is None or vo.poll() is None:
    time.sleep(120)
    print('ACOUSTIC:', subprocess.getoutput(f'tail -n 1 {BASE}/ac.log')[:150])
    print('VOCODER :', subprocess.getoutput(f'tail -n 1 {BASE}/voc.log')[:150])
print('done', ac.returncode, vo.returncode)

If a log line shows `nan`, tell me before continuing.

In [ ]:
import os
BASE = '/content/ultron'   # (re)defined here so this cell works even after a runtime restart
assert os.path.exists(f'{BASE}/vtts'), 'Project files are gone (runtime restarted) - run cell 1 again and upload the zip.'
import sys; sys.path.insert(0, BASE)
from vtts.synth import Synthesizer
from vtts.audio import save_wav
from IPython.display import Audio, display
s = Synthesizer(f'{BASE}/runs/ultron/acoustic.pt', f'{BASE}/runs/ultron/vocoder.pt')
text = "I am Ultron, I come for peace and I want the Avengers extinction"
for spk, name in enumerate(s.speakers):
    y = s.tts(text, speaker=spk, speed=0.85)
    print(name); display(Audio(y, rate=s.audio.sr))

### Last cell: download both (do this before closing!)
`ultron_models.zip` = the two files you use on your PC. `ultron_project_next.zip` = what you upload next session to continue.

In [ ]:
import os
BASE = '/content/ultron'   # (re)defined here so this cell works even after a runtime restart
assert os.path.exists(f'{BASE}/vtts'), 'Project files are gone (runtime restarted) - run cell 1 again and upload the zip.'
!cd {BASE}/runs/ultron && zip -q -j /content/ultron_models.zip acoustic.pt vocoder.pt
!cd {BASE} && zip -q -r /content/ultron_project_next.zip vtts data/all runs/ultron/acoustic_last.pt runs/ultron/vocoder_last.pt
from google.colab import files
files.download('/content/ultron_models.zip')
files.download('/content/ultron_project_next.zip')